# Uji signifikansi, bootstrap, Holm, dan besar efek

[![Buka di Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/imamrozi/recsys-hibrida-multisinyal-notebooks/blob/main/notebooks/bab06_evaluasi/03_uji_signifikansi_dan_koreksi_holm.ipynb)

**Buku pendamping:** *Sistem Rekomendasi Hibrida Multi-Sinyal: Rating, Sentimen, dan Emosi* · Bab 6 · Kode 6.x (uji berpasangan)

**Tujuan belajar**

- Menjalankan uji berpasangan (Wilcoxon, uji-t), bootstrap CI, dan koreksi Holm.
- Melihat bahwa p kecil tidak berarti efek besar (d_z) dan sebaliknya.
- Mengamati inflasi positif palsu pada perbandingan jamak tanpa koreksi.

> **Catatan jujur.** Semua data di notebook ini **simulasi** dan dibuat agar mudah dijalankan (CPU, hitungan detik). Angka yang keluar menunjukkan *perilaku metode*, bukan hasil empiris pada dataset nyata, dan tidak boleh dikutip sebagai bukti. Kode bersifat ilustratif dan **belum production-ready** (tanpa validasi masukan, tanpa uji beban).

## 1. Kode dari buku

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats


def paired_tests(m_a, m_b):
    d = np.asarray(m_b, float) - np.asarray(m_a, float)
    nz = d[d != 0]  # Wilcoxon mengabaikan selisih nol
    p_w = stats.wilcoxon(nz).pvalue if len(nz) > 0 else 1.0
    p_t = stats.ttest_rel(m_b, m_a).pvalue
    return {"mean_diff": d.mean(), "p_wilcoxon": p_w, "p_ttest": p_t,
            "d_z": d.mean() / d.std(ddof=1),
            "n_better": int((d > 0).sum()), "n_worse": int((d < 0).sum())}


def bootstrap_ci(m_a, m_b, n_boot=2000, alpha=0.05, seed=0):
    d = np.asarray(m_b, float) - np.asarray(m_a, float)
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(d), size=(n_boot, len(d)))
    means = d[idx].mean(axis=1)
    return np.percentile(means, [100 * alpha / 2, 100 * (1 - alpha / 2)])


def holm(pvalues, alpha=0.05):
    """True bila hipotesis nol ditolak. Urutan hasil sama dengan urutan masukan."""
    p = np.asarray(pvalues)
    m = len(p)
    reject = np.zeros(m, bool)
    for j, idx in enumerate(np.argsort(p)):
        if p[idx] <= alpha / (m - j):
            reject[idx] = True
        else:
            break
    return reject.tolist()

## 2. Skor per pengguna

Metrik per pengguna (misalnya nDCG@10) biasanya tidak normal: banyak nol, ekor panjang. Kita buat dua model dengan skor per pengguna dari campuran nol dan Beta, di mana B memiliki sedikit perbaikan nyata.

In [2]:
rng = np.random.default_rng(0)

def skor_pengguna(n, rata, p_nol=0.5, rng=rng):
    """Campuran: p_nol pengguna dapat skor 0; sisanya Beta (rata-rata sekitar `rata`). Rata-rata keseluruhan lebih rendah karena nol."""
    nol = rng.random(n) < p_nol
    b = rng.beta(2, 2 / rata - 2 + 1e-9, n) if rata < 1 else rng.beta(2, 2, n)
    return np.where(nol, 0.0, b)

def varian(base, efek, r):
    """Model B = A + efek + derau per pengguna. Pengguna dengan skor 0 tetap 0 (tidak ada yang bisa
    diperbaiki), dan skor di-clip ke [0, 1]. Penanganan nol ini penting: meng-clip nol akan
    membuat B > A secara artifisial walau efek = 0."""
    return np.where(base > 0, np.clip(base + r.normal(efek, 0.10, len(base)), 0, 1), 0.0)

n = 500
A = skor_pengguna(n, 0.30)
B = varian(A, 0.02, rng)
print("rata-rata A:", A.mean().round(4), "| B:", B.mean().round(4), "| proporsi skor nol A:", (A == 0).mean().round(2))
hasil = paired_tests(A, B)
ci = bootstrap_ci(A, B)
print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in hasil.items()})
print("CI bootstrap 95% untuk selisih rata-rata:", ci.round(4))

rata-rata A: 0.1666 | B: 0.1763 | proporsi skor nol A: 0.45
{'mean_diff': np.float64(0.0097), 'p_wilcoxon': np.float64(0.0031), 'p_ttest': np.float64(0.0025), 'd_z': np.float64(0.1361), 'n_better': 149, 'n_worse': 128}
CI bootstrap 95% untuk selisih rata-rata: [0.0036 0.0157]


## 3. p kecil ≠ efek besar

Selisih sebenarnya sama-sama kecil. Perbesar `n` dan lihat `p` serta `d_z` bergerak berbeda: `p` menyusut cepat terhadap `n`, sedangkan `d_z` mencerminkan efek per pengguna dan relatif stabil.

In [3]:
rows = []
for n_ in (50, 200, 1000, 5000, 20000):
    r_ = np.random.default_rng(1)
    a_ = skor_pengguna(n_, 0.30, rng=r_)
    b_ = varian(a_, 0.01, r_)                                 # efek sejati kecil: 0.01
    h = paired_tests(a_, b_)
    rows.append({"n": n_, "selisih": h["mean_diff"], "p_t": h["p_ttest"], "p_wilcoxon": h["p_wilcoxon"], "d_z": h["d_z"]})
skala = pd.DataFrame(rows).set_index("n")
print(skala.round(4))
assert skala["p_t"].iloc[-1] < skala["p_t"].iloc[0]
assert skala["d_z"].abs().max() < 0.5          # efek sebenarnya tetap kecil di semua n

       selisih     p_t  p_wilcoxon     d_z
n                                         
50      0.0093  0.2661      0.2016  0.1591
200     0.0088  0.0893      0.0738  0.1207
1000    0.0056  0.0105      0.0201  0.0810
5000    0.0061  0.0000      0.0000  0.0898
20000   0.0066  0.0000      0.0000  0.0960


**Interpretasi.** Dengan data cukup banyak, efek yang tak berarti secara praktis pun menjadi "signifikan". Itu sebabnya buku merekomendasikan melaporkan selisih rata-rata dengan CI dan `d_z`, bukan `p` saja. Dalam simulasi ini, `d_z` tetap kecil (di bawah 0,5) pada semua `n`, sedangkan `p` menyusut. Perhatikan juga bahwa pada `n` kecil, `p` besar belum berarti tidak ada efek: itu berarti datanya kurang kuat untuk memutuskan (daya uji rendah).

## 4. Perbandingan jamak: inflasi positif palsu dan Holm

Bayangkan memperbandingkan 20 varian model terhadap baseline. Pada simulasi ini **tidak ada** varian yang lebih baik (hipotesis nol benar untuk semuanya). Berapa varian yang "signifikan" pada α = 0,05 tanpa dan dengan Holm? Ulangi 200 percobaan.

In [4]:
def satu_percobaan(seed, m=20, n=300):
    r = np.random.default_rng(seed)
    base = skor_pengguna(n, 0.30, rng=r)
    p = []
    for _ in range(m):
        v = varian(base, 0.0, r)                                    # tidak ada efek sejati
        p.append(paired_tests(base, v)["p_wilcoxon"])
    return np.array(p)

mentah, terkoreksi = [], []
for s in range(200):
    p = satu_percobaan(s)
    mentah.append((p < 0.05).any()); terkoreksi.append(any(holm(p)))
fwer_mentah, fwer_holm = np.mean(mentah), np.mean(terkoreksi)
print(f"peluang >=1 positif palsu: tanpa koreksi {fwer_mentah:.2f} | dengan Holm {fwer_holm:.2f}")
print(f"teori tanpa koreksi: 1-(0.95)^20 = {1 - 0.95**20:.2f}")
assert fwer_mentah > 0.5
assert fwer_holm < 0.12

peluang >=1 positif palsu: tanpa koreksi 0.71 | dengan Holm 0.07
teori tanpa koreksi: 1-(0.95)^20 = 0.64


**Interpretasi.** Menjalankan 20 pembandingan pada α = 0,05 hampir pasti menghasilkan setidaknya satu "temuan" palsu (teori: ~64%). Holm menekan peluang itu mendekati α (family-wise error rate) tanpa mengasumsikan independensi. Dalam penelitian nyata, jumlah perbandingan yang diuji, termasuk yang dibuang, harus dilaporkan. Catatan: simulasi memakai derau independen antar varian; pada varian yang saling berkorelasi, perilakunya berbeda.

## 5. Daya uji: seberapa sering efek nyata terdeteksi?

In [5]:
def daya(efek, n, ulang=200):
    det = 0
    for s in range(ulang):
        r = np.random.default_rng(1000 + s)
        a_ = skor_pengguna(n, 0.30, rng=r); b_ = varian(a_, efek, r)
        det += paired_tests(a_, b_)["p_wilcoxon"] < 0.05
    return det / ulang

grid = pd.DataFrame({f"efek={e}": [daya(e, n_) for n_ in (50, 200, 800)] for e in (0.0, 0.01, 0.03)}, index=(50, 200, 800))
grid.index.name = "n"
print(grid.round(2))
assert grid["efek=0.0"].max() < 0.12            # tanpa efek: ~alpha
assert grid["efek=0.03"].iloc[-1] > grid["efek=0.03"].iloc[0]

     efek=0.0  efek=0.01  efek=0.03
n                                  
50       0.04       0.08       0.33
200      0.07       0.19       0.84
800      0.05       0.64       1.00


**Interpretasi.** Kolom `efek=0.0` memperlihatkan tingkat positif palsu uji tunggal: sekitar α. Untuk efek kecil, `n` kecil hampir tidak pernah mendeteksinya. Jadi hasil "tidak signifikan" pada sampel kecil bukan bukti ketiadaan efek.

## Pertanyaan refleksi

1. Mengapa Wilcoxon sering dipilih untuk metrik per pengguna? Kapan uji-t sah walau data tidak normal?
2. Ubah `skor_pengguna` menjadi tanpa nol (`p_nol=0`). Bagaimana hasil Wilcoxon vs uji-t berubah?
3. Pada Bagian 4, buat varian yang saling berkorelasi (derau bersama). Bagaimana FWER tanpa koreksi berubah?